# UAVDT Annotation Test Notebook

This notebook validates UAVDT ground-truth annotations for `M1401_gt_whole.txt` by rendering original video frames with labeled bounding boxes and `target_id` values.

Expected local inputs:

- Image sequence folder, for example: `UAV_benchmark_M/M1401`
- Annotation file, for example: `local_uav_bev_project/annotations/M1401_gt_whole.txt`

The output is a set of annotated frames and an MP4 preview video saved under the local project workspace.


In [ ]:
#@title 1. Set local project paths and create output folders

from pathlib import Path
import os
from notebook_local import resolve_project_dir, print_local_setup

PROJECT_DIR = resolve_project_dir()
ANNOTATIONS_DIR = PROJECT_DIR / 'annotations'
OUTPUT_DIR = PROJECT_DIR / 'work' / 'notebook_gt_annotation_test'
FRAMES_OUT_DIR = OUTPUT_DIR / 'annotated_frames'

ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_OUT_DIR.mkdir(parents=True, exist_ok=True)

print_local_setup(PROJECT_DIR)
print('ANNOTATIONS_DIR:', ANNOTATIONS_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('FRAMES_OUT_DIR:', FRAMES_OUT_DIR)


## Upload note

Place `M1401_gt_whole.txt` in one of these locations, or edit `GT_WHOLE_PATH` in the next cell:

```text
local_uav_bev_project/annotations/M1401_gt_whole.txt
local_uav_bev_project/GT/M1401_gt_whole.txt
local_uav_bev_project/raw/UAV-benchmark-MOTD_v1.0/GT/M1401_gt_whole.txt
```


In [ ]:
#@title 2. Configure paths and frame range
from pathlib import Path
from notebook_local import default_gt_path, find_sequence_dir

# Image folder containing img000001.jpg, img000002.jpg, ...
IMAGE_DIR = find_sequence_dir('M1401', PROJECT_DIR) #@param {type:'string'}
IMAGE_DIR = Path(IMAGE_DIR)

# Path to the UAVDT gt_whole annotation file
GT_WHOLE_PATH = default_gt_path('M1401_gt_whole.txt', PROJECT_DIR) #@param {type:'string'}
GT_WHOLE_PATH = Path(GT_WHOLE_PATH)

START_FRAME_INDEX = 1 #@param {type:'integer'}
NUM_FRAMES_TO_RENDER = 30 #@param {type:'integer'}
FRAME_STEP = 10 #@param {type:'integer'}
BOX_THICKNESS = 2 #@param {type:'integer'}
FONT_SCALE = 0.5 #@param {type:'number'}


In [ ]:
#@title 3. Find annotation file if needed

from pathlib import Path
from notebook_local import default_gt_path

candidate_paths = [
    GT_WHOLE_PATH,
    PROJECT_DIR / 'annotations' / 'M1401_gt_whole.txt',
    PROJECT_DIR / 'GT' / 'M1401_gt_whole.txt',
    PROJECT_DIR / 'raw' / 'UAV-benchmark-MOTD_v1.0' / 'GT' / 'M1401_gt_whole.txt',
    PROJECT_DIR / 'raw' / 'UAV_benchmark_MOTD_v1.0' / 'GT' / 'M1401_gt_whole.txt',
    default_gt_path('M1401_gt_whole.txt', PROJECT_DIR),
]

found = None
for p in candidate_paths:
    p = Path(p)
    print('Candidate:', p, 'exists:', p.exists())
    if p.exists() and found is None:
        found = p

if found is None:
    print('Searching PROJECT_DIR for M1401_gt_whole.txt. This may take a moment.')
    matches = list(PROJECT_DIR.rglob('M1401_gt_whole.txt'))
    print('Matches found:', len(matches))
    for p in matches[:20]:
        print(' ', p)
    if matches:
        found = matches[0]

if found is None:
    raise FileNotFoundError('Could not find M1401_gt_whole.txt locally. Put it under annotations/, GT/, or raw/.../GT and rerun this cell.')

GT_WHOLE_PATH = found
print('Using annotation file:', GT_WHOLE_PATH)


In [ ]:
#@title 4. Load UAVDT M1401_gt_whole annotations

import pandas as pd
import numpy as np

cols = [
    'frame_index',
    'target_id',
    'bbox_left',
    'bbox_top',
    'bbox_width',
    'bbox_height',
    'out_of_view',
    'occlusion',
    'object_category',
]

ann = pd.read_csv(GT_WHOLE_PATH, header=None, names=cols)

for c in cols:
    ann[c] = pd.to_numeric(ann[c], errors='coerce')

ann = ann.dropna(subset=['frame_index', 'target_id', 'bbox_left', 'bbox_top', 'bbox_width', 'bbox_height']).copy()
ann['frame_index'] = ann['frame_index'].astype(int)
ann['target_id'] = ann['target_id'].astype(int)
ann['object_category'] = ann['object_category'].fillna(0).astype(int)

ann['x1'] = ann['bbox_left'].astype(float)
ann['y1'] = ann['bbox_top'].astype(float)
ann['x2'] = ann['bbox_left'].astype(float) + ann['bbox_width'].astype(float)
ann['y2'] = ann['bbox_top'].astype(float) + ann['bbox_height'].astype(float)
ann['box_area'] = ann['bbox_width'].astype(float) * ann['bbox_height'].astype(float)
ann['track_id'] = ann['target_id']

category_map = {1: 'car', 2: 'truck', 3: 'bus'}
ann['class_name'] = ann['object_category'].map(category_map).fillna('vehicle')

print('Rows:', len(ann))
print('Frame range:', int(ann['frame_index'].min()), 'to', int(ann['frame_index'].max()))
print('Unique target IDs:', ann['target_id'].nunique())
print('Category counts:')
print(ann['class_name'].value_counts())
print('First rows:')
display(ann.head())


In [ ]:
#@title 5. Verify image naming and frame-index mapping

from pathlib import Path
import re

if not IMAGE_DIR.exists():
    raise FileNotFoundError('IMAGE_DIR does not exist: ' + str(IMAGE_DIR))

image_files = sorted([p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in ['.jpg', '.jpeg', '.png']])
print('Images found:', len(image_files))
print('First images:')
for p in image_files[:10]:
    print(' ', p.name)
print('Last images:')
for p in image_files[-5:]:
    print(' ', p.name)

# Build a mapping from numeric image index to path, supporting img000001.jpg style names.
image_index_to_path = {}
for p in image_files:
    m = re.search(r'(\d+)', p.stem)
    if m:
        image_index_to_path[int(m.group(1))] = p

print('Indexed image filenames:', len(image_index_to_path))
for idx in [1, 2, 10, START_FRAME_INDEX]:
    print('Image index', idx, 'exists:', idx in image_index_to_path, image_index_to_path.get(idx))

# If frame_index=1 exists as img000001.jpg, no offset is needed.
# The notebook uses original UAVDT frame_index directly as image index.


In [ ]:
#@title 6. Filter annotations for selected frames

selected_frame_indices = [START_FRAME_INDEX + i * FRAME_STEP for i in range(NUM_FRAMES_TO_RENDER)]
selected_frame_indices = [idx for idx in selected_frame_indices if idx in image_index_to_path]

if not selected_frame_indices:
    raise RuntimeError('No selected frames exist in IMAGE_DIR. Check START_FRAME_INDEX, FRAME_STEP, and image names.')

ann_sel = ann[ann['frame_index'].isin(selected_frame_indices)].copy()
ann_sel = ann_sel[ann_sel['box_area'] >= MIN_BOX_AREA].copy()
ann_sel = ann_sel[ann_sel['out_of_view'] <= MAX_OUT_OF_VIEW].copy()
if not KEEP_OCCLUDED:
    ann_sel = ann_sel[ann_sel['occlusion'] == 0].copy()

ann_sel = ann_sel.sort_values(['frame_index', 'track_id']).reset_index(drop=True)

sample_ann_path = OUTPUT_DIR / 'M1401_selected_annotations.csv'
ann_sel.to_csv(sample_ann_path, index=False)

print('Selected frames:', len(selected_frame_indices))
print('First selected frames:', selected_frame_indices[:20])
print('Selected annotation rows:', len(ann_sel))
print('Selected unique tracks:', ann_sel['track_id'].nunique())
print('Saved selected annotations:', sample_ann_path)
display(ann_sel.head())


In [ ]:
#@title 7. Render annotated frames

import cv2
import numpy as np
from PIL import Image

# Deterministic color per track ID
def color_for_id(track_id):
    rng = np.random.default_rng(int(track_id) * 1009 + 17)
    color = rng.integers(40, 255, size=3).tolist()
    return int(color[0]), int(color[1]), int(color[2])

CLASS_SHORT = {'car': 'car', 'truck': 'trk', 'bus': 'bus', 'vehicle': 'veh'}

rendered_paths = []

for frame_index in selected_frame_indices:
    img_path = image_index_to_path[frame_index]
    img = cv2.imread(str(img_path))
    if img is None:
        print('Could not read image:', img_path)
        continue

    fdf = ann_sel[ann_sel['frame_index'] == frame_index]
    for _, r in fdf.iterrows():
        x1 = int(round(r['x1']))
        y1 = int(round(r['y1']))
        x2 = int(round(r['x2']))
        y2 = int(round(r['y2']))
        track_id = int(r['track_id'])
        class_name = str(r['class_name'])
        color = color_for_id(track_id)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = f"{track_id}:{CLASS_SHORT.get(class_name, class_name)}"
        label_y = max(14, y1 - 5)
        cv2.putText(img, label, (x1, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)

    cv2.putText(img, f"frame_index={frame_index}  file={img_path.name}  boxes={len(fdf)}", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

    out_path = FRAMES_OUT_DIR / f'annotated_{frame_index:06d}.jpg'
    cv2.imwrite(str(out_path), img)
    rendered_paths.append(out_path)

print('Rendered frames:', len(rendered_paths))
print('First rendered frame:', rendered_paths[0] if rendered_paths else None)


In [ ]:
#@title 8. Display a few rendered examples

import matplotlib.pyplot as plt
import cv2

SHOW_COUNT = 4 #@param {type:'integer'}

for p in rendered_paths[:SHOW_COUNT]:
    img = cv2.imread(str(p))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    fig_w = 10
    fig_h = fig_w * h / w
    plt.figure(figsize=(fig_w, fig_h))
    plt.imshow(img_rgb)
    plt.title(p.name)
    plt.axis('off')
    plt.show()


In [ ]:
#@title 9. Save annotated video

import imageio.v2 as imageio

VIDEO_FPS = 10 #@param {type:'integer'}
video_path = OUTPUT_DIR / 'M1401_gt_whole_annotation_preview.mp4'
gif_path = OUTPUT_DIR / 'M1401_gt_whole_annotation_preview.gif'

if not rendered_paths:
    raise RuntimeError('No rendered frames available. Run Cell 7 first.')

frames = []
for p in rendered_paths:
    img = imageio.imread(p)
    frames.append(img)

imageio.mimsave(video_path, frames, fps=VIDEO_FPS)
imageio.mimsave(gif_path, frames, fps=VIDEO_FPS)

print('Saved video:', video_path)
print('Saved gif:', gif_path)


In [ ]:
#@title 10. Track continuity quick check

track_summary = ann_sel.groupby('track_id').agg(
    first_frame=('frame_index', 'min'),
    last_frame=('frame_index', 'max'),
    n_boxes=('frame_index', 'count'),
    class_name=('class_name', lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
).reset_index()
track_summary['frame_span'] = track_summary['last_frame'] - track_summary['first_frame'] + 1
track_summary = track_summary.sort_values(['n_boxes', 'frame_span'], ascending=False)

summary_path = OUTPUT_DIR / 'M1401_selected_track_summary.csv'
track_summary.to_csv(summary_path, index=False)

print('Saved track summary:', summary_path)
print('Top tracks by number of selected boxes:')
display(track_summary.head(20))


## What to check visually

In the rendered video, verify:

1. Boxes match visible vehicles.
2. The same physical car keeps the same `target_id` across frames.
3. The selected frame indices match your intended original UAVDT frames.

If this looks correct, the next step is to update Notebook #3/#4 to use `M1401_gt_whole.txt` instead of YOLO + simple nearest-neighbor tracking.
